# Multiple Disease Prediction System

## Data Preprocessing — Diabetes Dataset

Goal:
Build a reusable preprocessing pipeline for deployment.

In [69]:
# Step 1 : Import Required Libraries
# Data Manipulation Libraries
import pandas as pd
import numpy as np
# Scikit-Learn Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.base import (
    BaseEstimator,
    TransformerMixin
)
# Save Pipeline
import pickle

In [70]:
# Load the dataset into a pandas DataFrame
df = pd.read_csv("/content/diabetes (3).csv")

# Display the first few rows to confirm successful loading (brief check, not EDA)
print("Dataset loaded successfully. Displaying first 5 rows:")
display(df.head())

Dataset loaded successfully. Displaying first 5 rows:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


Raw Dataset

↓

Replace Invalid Zero Values

↓

Train-Test Split

↓

Outlier Removal

↓

Median Imputation

↓

Feature Scaling

↓

ColumnTransformer

↓

Save Pipeline

In [71]:
# ==========================================================
# Step 3 : Dataset Information
# ==========================================================

print(f"Number of Rows    : {df.shape[0]}")

print(f"Number of Columns : {df.shape[1]}")

print("\nFeature Names\n")

print(df.columns.tolist())

Number of Rows    : 768
Number of Columns : 9

Feature Names

['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


In [72]:
# Step 4 : Define Features and Target
FEATURE_COLUMNS = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age"
]

TARGET_COLUMN = "Outcome"
# Separate Features and Target
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()
print("Features Shape :", X.shape)
print("Target Shape   :", y.shape)

Features Shape : (768, 8)
Target Shape   : (768,)


In [73]:
# Step 5 : Replace Impossible Zero Values
IMPOSSIBLE_ZERO_COLUMNS = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]
# Create a copy to avoid modifying the original dataset
X_processed = X.copy()
# Replace impossible zeros with NaN
for column in IMPOSSIBLE_ZERO_COLUMNS:
    X_processed[column] = X_processed[column].replace(0, np.nan)
print("Invalid Zero Values Successfully Replaced with NaN ✅")
display(
    X_processed[IMPOSSIBLE_ZERO_COLUMNS]
    .isnull()
    .sum()
    .to_frame("Missing Values")
)

Invalid Zero Values Successfully Replaced with NaN ✅


,Missing Values
Glucose,5
BloodPressure,35
SkinThickness,227
Insulin,374
BMI,11


In [74]:
# Step 6 : Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("Train-Test Split Completed Successfully ✅\n")
print("Training Samples :", X_train.shape)
print("Testing Samples  :", X_test.shape)

Train-Test Split Completed Successfully ✅

Training Samples : (614, 8)
Testing Samples  : (154, 8)


In [75]:
# Step 7 : Verify Stratification
print("Training Set Class Distribution\n")
display(
    y_train
    .value_counts(normalize=True)
    .to_frame("Percentage")
)
print("\nTesting Set Class Distribution\n")
display(
    y_test
    .value_counts(normalize=True)
    .to_frame("Percentage")
)

Training Set Class Distribution



,Percentage
Outcome,
0,0.651466
1,0.348534



Testing Set Class Distribution



,Percentage
Outcome,
0,0.649351
1,0.350649


In [ ]:
# Step 8 : Custom OutlierRemover Class
from src.outlier_remover import OutlierRemover

In [77]:
# Step 9 : Verify OutlierRemover
outlier_remover = OutlierRemover()
X_train_outliers = outlier_remover.fit_transform(X_train)
print("OutlierRemover executed successfully.\n")
print("Missing values after replacing outliers:\n")
display(
    X_train_outliers
    .isnull()
    .sum()
    .to_frame("NaN Count")
)

OutlierRemover executed successfully.

Missing values after replacing outliers:



,NaN Count
Pregnancies,2
Glucose,4
BloodPressure,36
SkinThickness,177
Insulin,307
BMI,15
DiabetesPedigreeFunction,19
Age,6


In [78]:
# Step 10 : Numerical Pipeline
NUMERICAL_PIPELINE = Pipeline(
    steps=[
        (
            "outlier_remover",
            OutlierRemover()
        ),
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)
print("Numerical Pipeline Created Successfully.")

Numerical Pipeline Created Successfully.


In [79]:
# Step 12 : Display Pipeline
print(NUMERICAL_PIPELINE)

Pipeline(steps=[('outlier_remover', OutlierRemover()),
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


In [80]:
# Step 13 : Create ColumnTransformer
PREPROCESSOR = ColumnTransformer(
    transformers=[
        (
            "numerical_features",
            NUMERICAL_PIPELINE,
            FEATURE_COLUMNS
        )
    ],
    remainder="drop"
)

print("ColumnTransformer Created Successfully.")

ColumnTransformer Created Successfully.


In [81]:
# Step 14 : Fit Preprocessor
PREPROCESSOR.fit(X_train)

print("✓ Preprocessor fitted successfully on training data.")

✓ Preprocessor fitted successfully on training data.


In [82]:
# Step 15 : Transform Data
X_train_processed = PREPROCESSOR.transform(X_train)

X_test_processed = PREPROCESSOR.transform(X_test)

print("✓ Training data transformed successfully.")

print("✓ Testing data transformed successfully.")

print("✓ No data leakage occurred.")

✓ Training data transformed successfully.
✓ Testing data transformed successfully.
✓ No data leakage occurred.


In [83]:
# Step 16 : Convert to DataFrame
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=FEATURE_COLUMNS,
    index=X_train.index
)
X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=FEATURE_COLUMNS,
    index=X_test.index
)

print("Processed DataFrames Created Successfully.")

Processed DataFrames Created Successfully.


In [84]:

# Step 17 : Display Processed Data
print("Processed Training Dataset")

display(X_train_processed.head())

print("Processed Testing Dataset")

display(X_test_processed.head())

Processed Training Dataset


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
353,-0.856875,-1.056427,-0.910109,-2.026318,-1.622986,-0.786774,0.549181,-0.799077
711,0.376703,0.144399,0.568705,-0.225451,-2.037262,-0.410226,-0.001833,0.628999
373,-0.548480,-0.556083,-1.279812,1.335301,-0.616885,0.421316,-0.838124,-0.709822
46,-0.856875,0.811525,-1.464664,0.014665,-0.103971,-0.394537,0.486654,-0.352803
682,-1.165269,-0.889646,-0.725257,1.215243,-0.399883,1.943196,-0.287110,-0.977586


Processed Testing Dataset


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
44,0.993492,1.245157,-0.725257,0.014665,-0.103971,-0.755395,-0.568478,0.628999
672,1.918676,-1.790266,0.014150,-0.705682,-1.504621,0.515453,-0.603650,1.253782
700,-0.548480,0.010974,0.383853,-0.225451,1.474226,0.578211,0.170115,-0.620568
630,0.993492,-0.255876,-0.725257,0.014665,-0.103971,-0.755395,1.143182,0.093470
81,-0.548480,-1.590128,0.014150,0.014665,-0.103971,0.029079,-1.318795,-0.977586


The preprocessing pipeline is fitted ONLY on training data.

This prevents Data Leakage.

In [85]:
# Step 18 : Verify Training Dataset
print("=" * 60)
print("Verification : Training Dataset")
print("=" * 60)
# Missing Values
print("\nMissing Values")

display(
    X_train_processed
    .isnull()
    .sum()
    .to_frame("Missing Values")
)
# Feature Means
print("\nFeature Means")

display(
    X_train_processed
    .mean()
    .round(3)
    .to_frame("Mean")
)
# Feature Standard Deviations
print("\nFeature Standard Deviations")
display(
    X_train_processed
    .std()
    .round(3)
    .to_frame("Standard Deviation")
)
# Dataset Shape
print("\nDataset Shape")

print(X_train_processed.shape)
# Sample Records
print("\nFirst Five Rows")
display(
    X_train_processed.head()
)

Verification : Training Dataset

Missing Values


,Missing Values
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0



Feature Means


,Mean
Pregnancies,0.0
Glucose,-0.0
BloodPressure,0.0
SkinThickness,0.0
Insulin,-0.0
BMI,0.0
DiabetesPedigreeFunction,-0.0
Age,-0.0



Feature Standard Deviations


,Standard Deviation
Pregnancies,1.001
Glucose,1.001
BloodPressure,1.001
SkinThickness,1.001
Insulin,1.001
BMI,1.001
DiabetesPedigreeFunction,1.001
Age,1.001



Dataset Shape
(614, 8)

First Five Rows


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
353,-0.856875,-1.056427,-0.910109,-2.026318,-1.622986,-0.786774,0.549181,-0.799077
711,0.376703,0.144399,0.568705,-0.225451,-2.037262,-0.410226,-0.001833,0.628999
373,-0.548480,-0.556083,-1.279812,1.335301,-0.616885,0.421316,-0.838124,-0.709822
46,-0.856875,0.811525,-1.464664,0.014665,-0.103971,-0.394537,0.486654,-0.352803
682,-1.165269,-0.889646,-0.725257,1.215243,-0.399883,1.943196,-0.287110,-0.977586


In [86]:
# Step 19 : Verify Testing Dataset
print("=" * 60)
print("Verification : Testing Dataset")
print("=" * 60)
print("\nMissing Values")
display(
    X_test_processed
    .isnull()
    .sum()
    .to_frame("Missing Values")
)
print("\nFeature Means")
display(
    X_test_processed
    .mean()
    .round(3)
    .to_frame("Mean")
)
print("\nFeature Standard Deviations")
display(
    X_test_processed
    .std()
    .round(3)
    .to_frame("Standard Deviation")
)
print("\nDataset Shape")

print(X_test_processed.shape)


print("\nFirst Five Rows")

display(

    X_test_processed.head()

)

Verification : Testing Dataset

Missing Values


,Missing Values
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0



Feature Means


,Mean
Pregnancies,0.006
Glucose,-0.002
BloodPressure,0.120
SkinThickness,0.033
Insulin,0.048
BMI,-0.005
DiabetesPedigreeFunction,-0.145
Age,-0.086



Feature Standard Deviations


,Standard Deviation
Pregnancies,1.043
Glucose,1.075
BloodPressure,1.019
SkinThickness,0.954
Insulin,1.050
BMI,1.029
DiabetesPedigreeFunction,0.906
Age,0.928



Dataset Shape
(154, 8)

First Five Rows


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
44,0.993492,1.245157,-0.725257,0.014665,-0.103971,-0.755395,-0.568478,0.628999
672,1.918676,-1.790266,0.014150,-0.705682,-1.504621,0.515453,-0.603650,1.253782
700,-0.548480,0.010974,0.383853,-0.225451,1.474226,0.578211,0.170115,-0.620568
630,0.993492,-0.255876,-0.725257,0.014665,-0.103971,-0.755395,1.143182,0.093470
81,-0.548480,-1.590128,0.014150,0.014665,-0.103971,0.029079,-1.318795,-0.977586


In [87]:
# Step 20 : Save Preprocessor
PREPROCESSOR_FILENAME = "preprocessor_v1.pkl"
with open(
    PREPROCESSOR_FILENAME,
    "wb"
) as file:
    pickle.dump(
        PREPROCESSOR,
        file
    )
print("✓ Preprocessing Pipeline Saved Successfully.")
print(f"\nFilename : {PREPROCESSOR_FILENAME}")

✓ Preprocessing Pipeline Saved Successfully.

Filename : preprocessor_v1.pkl


In [88]:
# Step 21 : Load Saved Pipeline
with open(
    PREPROCESSOR_FILENAME,
    "rb"
) as file:
    LOADED_PREPROCESSOR = pickle.load(file)
print("✓ Saved Pipeline Loaded Successfully.")
print(LOADED_PREPROCESSOR)

✓ Saved Pipeline Loaded Successfully.
ColumnTransformer(transformers=[('numerical_features',
                                 Pipeline(steps=[('outlier_remover',
                                                  OutlierRemover()),
                                                 ('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Pregnancies', 'Glucose', 'BloodPressure',
                                  'SkinThickness', 'Insulin', 'BMI',
                                  'DiabetesPedigreeFunction', 'Age'])])


In [89]:
# Step 22 : Test Saved Pipeline
sample_data = X_train.iloc[[0]]
sample_processed = LOADED_PREPROCESSOR.transform(sample_data)
print("✓ Sample transformed successfully.")
print("\nProcessed Sample Shape:")
print(sample_processed.shape)

✓ Sample transformed successfully.

Processed Sample Shape:
(1, 8)


Expected Output

✔ Missing values created

✔ Pipeline built

✔ Features scaled

✔ Pipeline saved

## Key Learnings

✔ Importance of preprocessing

✔ Data Leakage prevention

✔ Pipeline creation

✔ Custom Transformer

✔ ColumnTransformer

✔ Feature Scaling

✔ Reusable preprocessing